In [ ]:
import json
import pandas as pd
from pathlib import Path
from tabulate import tabulate


# Map evaluation folder name suffixes (set by exp_name in the run scripts) to
# paper-style display names. Each suffix appears between the experiment prefix
# (e.g. "4e5steps_") and the trailing "_<ode_t_steps>steps". Order matters:
# longer keys are checked first so "projection_all_gradient_guidance" wins
# over "projection_all".
DISPLAY_NAMES = [
    ("projection_all_gradient_guidance", "Projection-All + Gradient Guidance"),
    ("projection_late_gradient_guidance", "Projection-Late + Gradient Guidance"),
    ("projection_relaxed_gradient_guidance", "Projection-Relaxed + Gradient Guidance"),
    ("projection_all", "Projection-All"),
    ("projection_late", "Projection-Late"),
    ("projection_relaxed", "Projection-Relaxed"),
    ("hardflow_new", "HardFlow (l4casadi-free)"),
    ("hardflow", "HardFlow"),
    ("oc_flow", "OC-Flow"),
    ("gradient_guidance", "Gradient Guidance"),
    ("original", "Original"),
]


def get_display_name(experiment_name: str) -> str:
    for key, name in DISPLAY_NAMES:
        if key in experiment_name:
            return name
    return experiment_name


def analyze_burgers_results(env_name="burgers"):

    base_path = Path("../logs") / env_name / "eval"

    results = []

    for exp_dir in base_path.glob("*"):
        if not exp_dir.is_dir():
            continue

        results_file = exp_dir / "results.json"
        if not results_file.exists():
            continue

        with open(results_file, "r") as f:
            data = json.load(f)

        exp_name = exp_dir.name
        exp_name = exp_name.replace("4e5steps_", "")
        exp_name = exp_name.replace("_10steps", "")

        results.append(
            {
                "Experiment": get_display_name(exp_name),
                "J_energy_mean": data.get("control_energy_mean (J_energy)", None),
                "J_energy_std": data.get("control_energy_std", None),
                "R_p_controlled": data.get(
                    "point_exceed_ratio (R_p), controlled", None
                ),
                "R_t_controlled": data.get("time_exceed_ratio (R_t), controlled", None),
                "R_s_controlled": data.get(
                    "sample_exceed_ratio (R_s), controlled", None
                ),
                "R_p_predicted": data.get("point_exceed_ratio (R_p), predicted", None),
                "R_t_predicted": data.get("time_exceed_ratio (R_t), predicted", None),
                "R_s_predicted": data.get("sample_exceed_ratio (R_s), predicted", None),
                "Total Trials": data.get("num_samples", None),
                "Time_mean": data.get("computation_time_mean", None),
                "Time_std": data.get("computation_time_std", None),
            }
        )

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values("R_s_controlled", ascending=True)

    return df


def display_burgers_results(df):

    if df.empty:
        print("No results found in the burgers evaluation directory.")
        return

    table_data = []
    for _, row in df.iterrows():

        if pd.notna(row["J_energy_mean"]) and pd.notna(row["J_energy_std"]):
            j_energy_str = f"{row['J_energy_mean']:.6f}±{row['J_energy_std']:.6f}"
        elif pd.notna(row["J_energy_mean"]):
            j_energy_str = f"{row['J_energy_mean']:.6f}"
        else:
            j_energy_str = "N/A"

        if pd.notna(row["Time_mean"]) and pd.notna(row["Time_std"]):
            time_str = f"{row['Time_mean']:.3f}±{row['Time_std']:.3f}"
        elif pd.notna(row["Time_mean"]):
            time_str = f"{row['Time_mean']:.3f}"
        else:
            time_str = "N/A"

        table_data.append(
            [
                row["Experiment"],
                j_energy_str,
                (
                    f"{row['R_p_controlled']:.4f}"
                    if pd.notna(row["R_p_controlled"])
                    else "N/A"
                ),
                (
                    f"{row['R_t_controlled']:.4f}"
                    if pd.notna(row["R_t_controlled"])
                    else "N/A"
                ),
                (
                    f"{row['R_s_controlled']:.4f}"
                    if pd.notna(row["R_s_controlled"])
                    else "N/A"
                ),
                (
                    f"{row['R_p_predicted']:.4f}"
                    if pd.notna(row["R_p_predicted"])
                    else "N/A"
                ),
                (
                    f"{row['R_t_predicted']:.4f}"
                    if pd.notna(row["R_t_predicted"])
                    else "N/A"
                ),
                (
                    f"{row['R_s_predicted']:.4f}"
                    if pd.notna(row["R_s_predicted"])
                    else "N/A"
                ),
                time_str,
                (f"{row['Total Trials']}" if pd.notna(row["Total Trials"]) else "N/A"),
            ]
        )

    headers = [
        "Experiment",
        "J_energy",
        "R_p (Controlled)",
        "R_t (Controlled)",
        "R_s (Controlled)",
        "R_p (Predicted)",
        "R_t (Predicted)",
        "R_s (Predicted)",
        "Average Time (s)",
        "Total Trials",
    ]

    print("\n" + "=" * 100)
    print(f"Results for PDE Control".center(100))
    print("=" * 100)
    print(f"\nTotal experiments found: {len(df)}\n")
    print(
        tabulate(
            table_data,
            headers=headers,
            tablefmt="grid",
            stralign="left",
            disable_numparse=True,
        )
    )
    print("\nMetrics explanation:")
    print("- J_energy: Control energy per grid point (mean±std)")
    print("- R_p: proportion of grid points exceeding safety threshold")
    print("- R_t: proportion of time steps exceeding safety threshold")
    print("- R_s: proportion of samples exceeding safety threshold")
    print("- Average Time: Average per-sample policy computation time (seconds)")
    print("=" * 100)

    return df


env_name = "burgers"
results_df = analyze_burgers_results(env_name)
display_burgers_results(results_df)